# Q(V, φ) parametrization

Collected charge **Q** of ETL LGADs vs bias **V** and fluence **φ**. Data is read directly from `parametrization_data.xlsx` (one sheet per production).

Modules: `config.py` (settings) · `data_io.py` (Excel reader) · `model.py` (fits) · `plots.py` (ROOT plots).

**Model.** Each production is described by a single smooth surface

$$Q(V,\varphi) = p_0(\varphi) + p_1(\varphi)\,V + \exp\big[p_2(\varphi) + p_3(\varphi)\,V\big],\qquad p_k(\varphi)=a_k+\exp(b_k\varphi+c_k)$$

fitted as **one global simultaneous fit** to all pooled points at once (`model.fit_global`). No floor is imposed: `p0` is free, so the charge decays to the no-gain bulk value on its own at high fluence. `config.P0_VS_PHI` toggles whether `p0` has its own fluence law (default) or is a single constant. The per-fluence fits `Q(V)=p0+p1·V+exp(p2+p3·V)` are kept as diagnostics. Charge error is `config.ERR_Q` (default 1 fC, switchable to 2).

Kernel: **Python (tb_analysis2)**. Full method: `docs/method.pdf`.

In [1]:
import ROOT
%jsroot on
import importlib
import config, data_io, model, plots
for m in (config, data_io, model, plots):
    importlib.reload(m)

# charge error: default 1 fC, switchable to 2
config.set_charge_error(1.0)

sensors = data_io.load_sensors()
print(data_io.summary(sensors))
print('\ncharge error = +-%.1f fC | param_mode = %s' % (config.ERR_Q, config.PARAM_MODE))

[HPK]
   HPK6 S4-A                    setup=Beta phi=0.00e+00  n=10  V[100..190]  Q[7.55..40.32]
   HPK6 S2-B                    setup=Beta phi=0.00e+00  n=10  V[100..190]  Q[7.14..36.38]
   HPK2 W13-3 - 4e14            setup=Beta phi=4.00e+14  n= 7  V[150..375]  Q[3.00..29.10]
   HPK2 W4-25 - 8e14            setup=Beta phi=8.00e+14  n= 5  V[300..500]  Q[2.50..12.50]
   HPK6 S4-B - 1e15             setup=Beta phi=1.00e+15  n= 7  V[300..550]  Q[2.42..7.96]
   HPK6 S5-A - 1e15             setup=Beta phi=1.00e+15  n= 7  V[300..550]  Q[2.42..9.93]
   HPK6 S5-B - 1.5e15           setup=Beta phi=1.50e+15  n= 7  V[300..550]  Q[0.71..1.55]
   HPK6 S5-B - 1.5e15           setup=Beta phi=1.50e+15  n= 7  V[300..550]  Q[1.19..2.01]
[FBK-LF]
   FBK-LF W16-S13               setup=Beta phi=0.00e+00  n=10  V[120..210]  Q[8.00..30.60]
   FBK-LF W16-S52               setup=Beta phi=0.00e+00  n=10  V[120..210]  Q[7.90..31.01]
   FBK-LF W18-S2 - 1e15         setup=Beta phi=1.00e+15  n= 9  V[300..600]  Q[2

In [2]:
# fit every production (sheet)
# (1) per-fluence Q(V) fits -> parameters p0..p3 (diagnostics, see tables below)
results = model.fit_all(sensors, verbose=True)

# (2) THE analytic model: ONE global simultaneous fit per production
#     Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V ),  p_k(phi)=a_k+exp(b_k*phi+c_k)
surfaces = model.fit_all_global(sensors)
print('\nAnalytic global law per production:')
for name, surf in surfaces.items():
    print('  ', name, '->', surf.law_string())


=== HPK (Beta) — 5 fluence points [param_mode=interp] ===
    HPK_0.0e+00                n=20 chi2/ndf=  5.63/16 [root] p=['-1.694', '0.08425', '-4.936', '0.04275']
    HPK_4.0e+14                n= 7 chi2/ndf=  0.18/3  [root] p=['2.203', '0.000282', '-2.61', '0.01575']
    HPK_8.0e+14                n= 5 chi2/ndf=  0.06/1  [root] p=['-0.2589', '2.43e-05', '-1.344', '0.007795']
    HPK_1.0e+15                n=14 chi2/ndf=  3.73/10 [root] p=['2.095', '9.906e-08', '-6.03', '0.01449']
    HPK_1.5e+15                n=14 chi2/ndf=  0.74/10 [root] p=['0.8806', '1.505e-05', '-5.468', '0.009694']
  parametrization for HPK:
    p0(phi) = interp[-1.69@0.0e+00, 2.2@4.0e+14, -0.259@8.0e+14, 2.1@1.0e+15, 0.881@1.5e+15]
    p1(phi) = interp[0.0842@0.0e+00, 0.000282@4.0e+14, 2.43e-05@8.0e+14, 9.91e-08@1.0e+15, 1.5e-05@1.5e+15]
    p2(phi) = interp[-4.94@0.0e+00, -2.61@4.0e+14, -1.34@8.0e+14, -6.03@1.0e+15, -5.47@1.5e+15]
    p3(phi) = interp[0.0428@0.0e+00, 0.0158@4.0e+14, 0.00779@8.0e+14, 0.0145@

## Q(V) fits — pooled data + fit (open red = test beam, cross-check only)

In [3]:
for name, r in results.items():
    plots.plot_QV(sensors, r).Draw()

## Parameter laws p₁(φ), p₂(φ), p₃(φ)
The three fluence-dependent parameters of the global surface, each with its exponential law (blue) and the uncertainty band propagated from the global fit. The markers lie on the laws by construction — it is one simultaneous fit.

In [4]:
for name, surf in surfaces.items():
    plots.plot_param_laws(surf).Draw()

## The analytic model vs data
The single global surface `Q(V,φ) = p0 + p1(φ)·V + exp(p2(φ) + p3(φ)·V)` overlaid on the pooled data at each measured fluence — one formula describing every fluence.

In [5]:
for name, surf in surfaces.items():
    plots.plot_surface_vs_data(sensors, surf).Draw()

## Bias to reach the target charge, vs fluence

In [6]:
for name, surf in surfaces.items():            # one figure per sheet
    plots.plot_bias_for_target({name: surf}, Q_target=config.Q_TARGET, phi_hi=1.6e15).Draw()
print('Bias to reach Q = %.0f fC:' % config.Q_TARGET)
for name, surf in surfaces.items():
    row = {}
    for phi in surf.fluences:
        v = surf.V_for_Q(config.Q_TARGET, phi)
        row['%.1e' % phi] = 'n/a' if v != v else '%.0f V' % v
    print(' ', name, row)

Bias to reach Q = 8 fC:
  HPK {'0.0e+00': '106 V', '4.0e+14': '280 V', '8.0e+14': '449 V', '1.0e+15': '543 V', '1.5e+15': '780 V'}
  FBK-LF {'0.0e+00': '121 V', '1.0e+15': '441 V', '1.5e+15': '572 V'}


## The analytic surface Q(V,φ) as a 2-D map

In [7]:
tf2s = {name: surf.tf2() for name, surf in surfaces.items()}
c = ROOT.TCanvas('csurf', 'surfaces', 500 * len(tf2s), 450)
c.Divide(len(tf2s), 1)
for i, (name, f2) in enumerate(tf2s.items()):
    c.cd(i + 1); ROOT.gPad.SetRightMargin(0.16)
    f2.SetMaximum(20); f2.Draw('colz')
c.Draw()

## Operating point: charge obtainable & bias needed vs fluence (SEB-capped)

**Cell 1** — for each production: dashed = bias needed to reach the target charge (15 fC), capped at the SEB limit `V_SEB` (HPK 550 V, FBK 600 V); solid = charge actually obtainable = `min(target, Q(V_SEB, φ))` (flat at the target until the SEB knee, then the charge available at `V_SEB`).

**Cell 2** — fix a charge (e.g. 8 fC) and read off the bias needed at each fluence (flagged if it exceeds `V_SEB`).

In [8]:
# === CELL 1 — charge obtained & bias needed vs fluence, with the SEB cap ===
# Uses the global surface (computed above): one smooth trend, charge converges on
# its own to the free bulk term p0 (~0.3-0.5 fC) at high fluence (no floor imposed).
TARGET = config.PLATEAU_CHARGE            # 15 fC (solid-line plateau)
# V_SEB per production from config.V_SEB (HPK 550 V, FBK 600 V); override with vmax={...}
# one separate figure per sheet; add overlay='auto' to superimpose simulation/<prod>.csv
for name, surf in surfaces.items():
    plots.plot_charge_bias_vs_fluence({name: surf}, target=TARGET, vmax=None).Draw()

In [9]:
# === CELL 2 — fix a charge, get the bias needed at each fluence ===
Q_FIX = 8.0   # fC  (change me)

for name, surf in surfaces.items():            # one figure per sheet
    plots.plot_bias_for_target({name: surf}, Q_target=Q_FIX, phi_hi=1.6e15).Draw()

print('Bias needed to reach Q = %.1f fC (global surface):' % Q_FIX)
for name, surf in surfaces.items():
    vmax = plots._vmax_for(name, None)
    for phi in surf.fluences:
        v = surf.V_for_Q(Q_FIX, phi, v_hi=1200)
        if v != v:
            s = 'not reachable'
        else:
            s = '%.0f V' % v + ('  (> V_SEB = %.0f V!)' % vmax if v > vmax else '')
        print('  %-6s  phi=%.2e  ->  %s' % (name, phi, s))

Bias needed to reach Q = 8.0 fC (global surface):
  HPK     phi=0.00e+00  ->  106 V
  HPK     phi=4.00e+14  ->  280 V
  HPK     phi=8.00e+14  ->  449 V
  HPK     phi=1.00e+15  ->  543 V
  HPK     phi=1.50e+15  ->  780 V  (> V_SEB = 550 V!)
  FBK-LF  phi=0.00e+00  ->  121 V
  FBK-LF  phi=1.00e+15  ->  441 V
  FBK-LF  phi=1.50e+15  ->  572 V


## Parameters of the production
Per-fluence Q(V) fit parameters `p0..p3` (with errors and χ²/ndf) and the vs-fluence laws actually used. Also written to `parameters_<prod>.csv`.

In [10]:
# === Parameters of each production ===
# The global surface  Q(V,phi) = p0 + p1(phi)*V + exp(p2(phi) + p3(phi)*V):
#   p0,p1,p2,p3  values of the fitted laws at each measured fluence (+/- band)
#   (p0 is a law when config.P0_VS_PHI, else a single constant)
import pandas as pd
for name, surf in surfaces.items():
    print('=' * 72)
    print(name, '   (p0 %s)' % ('law(phi)' if not surf.p0_const else 'const'))
    df = pd.DataFrame({
        'fluence': surf.fluences,
        'p0': surf.p0_nodes, 'p0_err': surf.p0_err,
        'p1': surf.p1_nodes, 'p1_err': surf.p1_err,
        'p2': surf.p2_nodes, 'p2_err': surf.p2_err,
        'p3': surf.p3_nodes, 'p3_err': surf.p3_err,
    })
    display(df.round(5))
    print('analytic laws:')
    print(surf.law_string())
    df.to_csv('parameters_%s.csv' % model._short(name), index=False)
    print('-> saved parameters_%s.csv' % model._short(name))

HPK    (p0 law(phi))


,fluence,p0,p0_err,p1,p1_err,p2,p2_err,p3,p3_err
0,0.000000e+00,5.12656,0.83611,0.0,0.00526,-2.01759,0.39526,0.02895,0.00198
1,4.000000e+14,2.89187,1.00341,0.0,0.00526,-3.28764,0.65567,0.01756,0.00167
2,8.000000e+14,1.63129,1.33143,0.0,0.00526,-3.28764,0.65567,0.01146,0.00116
3,1.000000e+15,1.22520,1.48978,0.0,0.00526,-3.28764,0.65567,0.00957,0.00089
4,1.500000e+15,0.59896,1.84253,0.0,0.00526,-3.28764,0.65567,0.00678,0.00078


analytic laws:
Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +6.1854e-12 + exp(-1.4313e-15*phi +1.6344e+00)
       p1(phi) = +3.7975e-20 + exp(-4.3588e-14*phi -6.7203e+01)
       p2(phi) = -3.2876e+00 + exp(-6.9537e-14*phi +2.3906e-01)
       p3(phi) = +4.4332e-03 + exp(-1.5631e-15*phi -3.7082e+00)
       chi2/ndf = 26.9/48   RMS = 0.67 fC
-> saved parameters_HPK.csv
FBK-LF    (p0 law(phi))


,fluence,p0,p0_err,p1,p1_err,p2,p2_err,p3,p3_err
0,0.000000e+00,4.89649,2.15559,0.0,0.02204,-1.73111,0.75831,0.02369,0.00322
1,1.000000e+15,0.49749,2.43327,0.0,0.01060,-1.73111,0.87785,0.00850,0.00125
2,1.500000e+15,0.49749,2.85732,0.0,0.01060,-1.73111,0.87785,0.00655,0.00086


analytic laws:
Q(V,phi) = p0(phi) + p1(phi)*V + exp( p2(phi) + p3(phi)*V )
       p0(phi) = +4.9749e-01 + exp(-1.9798e-14*phi +1.4814e+00)
       p1(phi) = +2.3199e-15 + exp(-8.0080e-15*phi -2.2380e+01)
       p2(phi) = -1.7311e+00 + exp(-1.8169e-14*phi -1.8884e+01)
       p3(phi) = +5.0955e-03 + exp(-1.6971e-15*phi -3.9850e+00)
       chi2/ndf = 6.0/44   RMS = 0.33 fC
-> saved parameters_FBKLF.csv
